# Optional Vision 02 - Autoencoders in PyTorch

**Guiding question:** how small can a visual code become before reconstruction evidence visibly fails?

> **What you finished last time:** the CNN learned reusable local filters.
> **The gap this chapter closes:** encoder/decoder reconstruction, latent inspection, and compression evidence.
> **What you'll have by the end:** a trained convolutional autoencoder on local shapes and a measured latent-width comparison.
> **What this chapter is not:** a VAE; no distributional latent objective is claimed.


In [ ]:
# -- Setup and local image dataset ------------------------------------------------
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

torch.manual_seed(21)
np.random.seed(21)
plt.rcParams.update({"figure.facecolor": "#1a1a2e", "axes.facecolor": "#1a1a2e"})

def make_image(index, size=16):
    image = torch.zeros(size, size)
    label = index % 4
    offset = int(torch.randint(-2, 3, ()).item())
    if label == 0:
        image[3:13, 7 + offset:9 + offset] = 1
    elif label == 1:
        image[7 + offset:9 + offset, 3:13] = 1
    elif label == 2:
        coordinates = torch.arange(3, 13)
        image[coordinates, coordinates + offset] = 1
    else:
        image[4:12, 4:12] = 1
        image[6:10, 6:10] = 0
    image += 0.05 * torch.randn_like(image)
    return image.clamp(0, 1), label

examples = [make_image(index) for index in range(480)]
images = torch.stack([item[0] for item in examples]).unsqueeze(1)
labels = torch.tensor([item[1] for item in examples])
train_images, valid_images = images[:400], images[400:]
train_labels, valid_labels = labels[:400], labels[400:]
loader = DataLoader(TensorDataset(train_images, train_labels), batch_size=40, shuffle=True)
print(f"Local images: {tuple(images.shape)}")


## 1 - Reconstruction creates a bottleneck complaint

```mermaid
flowchart LR
    A["16 x 16 image"] --> E["encoder"]
    E --> Z["latent code"]
    Z --> D["decoder"]
    D --> R["16 x 16 reconstruction"]
    R --> L["pixel reconstruction loss"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Z fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style L fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Encoder, decoder, and explicit shape contract -------------------------------
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(8, latent_channels, 3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_channels, 8, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(8, 1, 4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, inputs):
        latent = self.encoder(inputs)
        return self.decoder(latent), latent

model = ConvAutoencoder(latent_channels=4)
reconstruction, latent = model(train_images[:5])
print(f"Input:          {tuple(train_images[:5].shape)}")
print(f"Latent:         {tuple(latent.shape)}")
print(f"Reconstruction: {tuple(reconstruction.shape)}")
assert reconstruction.shape == train_images[:5].shape


In [ ]:
# -- Train reconstruction from pixels --------------------------------------------
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
history = []
for epoch in range(35):
    epoch_losses = []
    for batch, _ in loader:
        reconstruction, _ = model(batch)
        loss = nn.functional.mse_loss(reconstruction, batch)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        epoch_losses.append(float(loss.detach()))
    history.append(np.mean(epoch_losses))

with torch.inference_mode():
    valid_reconstruction, valid_latent = model(valid_images)
    valid_mse = nn.functional.mse_loss(valid_reconstruction, valid_images).item()
print(f"Training MSE: {history[0]:.6f} -> {history[-1]:.6f}")
print(f"Validation MSE: {valid_mse:.6f}")
assert history[-1] < history[0]


In [ ]:
# -- Original versus reconstructed evidence --------------------------------------
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for index in range(8):
    axes[0, index].imshow(valid_images[index, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, index].imshow(valid_reconstruction[index, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, index].axis("off")
    axes[1, index].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Rebuilt")
plt.show()


## 2 - The latent code is compressed evidence, not a guaranteed concept list

**Predict:** if the latent tensor is one quarter the input size, does that prove every latent channel maps to a human concept? No. Reconstruction pressure can organize useful features without naming them.


In [ ]:
# -- Latent-space inspection and compression rate --------------------------------
input_values = int(np.prod(valid_images.shape[1:]))
latent_values = int(np.prod(valid_latent.shape[1:]))
compression_ratio = input_values / latent_values
class_centers = []
flattened_latent = valid_latent.flatten(1)
for label in range(4):
    class_centers.append(flattened_latent[valid_labels == label].mean(dim=0))
center_distances = torch.cdist(torch.stack(class_centers), torch.stack(class_centers))

print(f"Input values per image: {input_values}")
print(f"Latent values per image: {latent_values}")
print(f"Compression ratio: {compression_ratio:.1f}x")
print("Distances between class-average latent codes:")
print(center_distances.round(decimals=2))


## 3 - Change latent width and measure the trade-off

```mermaid
flowchart LR
    S["smaller latent"] --> C["more compression"]
    C --> F["possible reconstruction failure"]
    L["larger latent"] --> R["less compression"]
    R --> Q["possible lower reconstruction error"]
    style S fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style L fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Your turn: one controlled latent-width comparison ---------------------------
def train_width(latent_channels, epochs=18):
    torch.manual_seed(21)
    candidate = ConvAutoencoder(latent_channels)
    optimizer = torch.optim.Adam(candidate.parameters(), lr=0.01)
    for _ in range(epochs):
        for batch, _ in loader:
            rebuilt, _ = candidate(batch)
            loss = nn.functional.mse_loss(rebuilt, batch)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
    with torch.inference_mode():
        rebuilt, latent_code = candidate(valid_images)
    return (
        nn.functional.mse_loss(rebuilt, valid_images).item(),
        input_values / int(np.prod(latent_code.shape[1:])),
    )

# CHANGE THIS: add 8 to compare a wider latent code
widths = [2, 4]
results = {width: train_width(width) for width in widths}
for width, (mse, ratio) in results.items():
    print(f"latent channels={width}: validation MSE={mse:.6f}, compression={ratio:.1f}x")
print("Check both reconstruction error and compression; neither metric owns the decision alone.")


## 4 - Coverage ledger

**Built and measured:** encoder, decoder, reconstruction, latent tensors, class-center geometry, compression, latent-width trade-off.

**Explained or illustrated:** a useful latent code need not have human-named axes.

**Named and out of scope:** variational autoencoders. A VAE needs a distributional latent objective and sampling evidence; this notebook intentionally does not fake that lab.
